# Thermal Human Shape Detection — Training Notebook

**Project Pi — Emergency Alert System**

Trains a lightweight CNN (32x24 thermal input) that distinguishes human-shaped thermal signatures from hot objects and background. Exports `thermal_human_shape.tflite` for deployment on Raspberry Pi.

---
## How to Run

1. **Click `Runtime → Run All`** — it is now safe to do so.
2. Cell 0 verifies the environment (no install, no restart needed).
3. Training takes ~5–10 min on Colab CPU, ~2 min on T4 GPU.
4. At the end, three files will auto-download: `.tflite`, `_labels.json`, `eval_metrics.json`.


## Part 0 — Environment Check

In [ ]:
# Cell 0 — Environment check
# Colab already ships TF, NumPy, sklearn, scipy, matplotlib, pandas, tqdm.
# This cell just verifies everything is present. No reinstall needed.
# Safe to re-run after any restart.
import sys, importlib, subprocess
print("Python:", sys.version)

# Install tflite-runtime if missing (optional, only needed for verify cell)
for pkg, imp in [("tqdm","tqdm"), ("scipy","scipy")]:
    if importlib.util.find_spec(imp) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
        print(f"Installed {pkg}")

import numpy as np
import tensorflow as tf
import sklearn, scipy, matplotlib
print("NumPy      :", np.__version__)
print("TensorFlow :", tf.__version__)
print("scikit-learn:", sklearn.__version__)
print("scipy      :", scipy.__version__)
print("matplotlib :", matplotlib.__version__)
print()
print("All dependencies ready. Proceed to Part 1 — no restart needed.")


---
## Part 1 — Imports and Configuration

In [ ]:
# Part 1 — Imports and Configuration (start here after restart)
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json, os, random, warnings
from pathlib import Path
from scipy.ndimage import zoom
warnings.filterwarnings("ignore")

WORK_DIR  = Path("/content/project_pi_thermal_shape")
DATA_DIR  = WORK_DIR / "data"
MODEL_DIR = WORK_DIR / "model"
PLOT_DIR  = WORK_DIR / "plots"
for d in [WORK_DIR, DATA_DIR, MODEL_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FRAME_H, FRAME_W = 24, 32
CLASS_NAMES = [
    "human_full","human_partial","human_head","human_torso",
    "human_hands","human_feet","hot_object","background","ambiguous"
]
NUM_CLASSES = len(CLASS_NAMES)
HUMAN_CLASS_IDS = set(range(6))
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED); tf.random.set_seed(RANDOM_SEED)

print("NumPy      :", np.__version__)
print("TensorFlow :", tf.__version__)
print("Python     :", os.sys.version)
print("GPU        :", tf.config.list_physical_devices("GPU") or "CPU only")
print("Classes    :", NUM_CLASSES)


---
## Part 2 — Synthetic Data Generation

In [ ]:
# Part 2 — Drawing primitives
def _bg(): return float(np.random.uniform(24.0, 28.0))
def _noise(shape, s=0.4): return np.random.normal(0, s, shape).astype(np.float32)
def new_frame(bg=None):
    bg = bg if bg is not None else _bg()
    return np.full((FRAME_H, FRAME_W), bg, dtype=np.float32)

def draw_ellipse(frm, cy, cx, ry, rx, temp, angle=0.0):
    H, W = frm.shape
    rows, cols = np.ogrid[:H, :W]
    ca = np.cos(np.radians(angle)); sa = np.sin(np.radians(angle))
    dr = rows - cy; dc = cols - cx
    rr =  ca*dr + sa*dc
    rc = -sa*dr + ca*dc
    ry = max(ry, 0.5); rx = max(rx, 0.5)
    mask = (rr/ry)**2 + (rc/rx)**2 <= 1.0
    frm[mask] = np.maximum(frm[mask], temp)

def draw_rect(frm, r0, c0, r1, c1, temp):
    r0i = max(0, int(r0)); r1i = min(frm.shape[0], int(r1)+1)
    c0i = max(0, int(c0)); c1i = min(frm.shape[1], int(c1)+1)
    if r0i < r1i and c0i < c1i:
        frm[r0i:r1i, c0i:c1i] = np.maximum(frm[r0i:r1i, c0i:c1i], temp)

print("Primitives ready.")


In [ ]:
# Part 2 — Human shape generators
def make_human_full():
    frm = new_frame(); sc = np.random.uniform(0.7, 1.2)
    pose = random.choice(["stand","sit","lie","arms_up"])
    if pose == "lie":
        cx = FRAME_W/2 + np.random.uniform(-3,3)
        cy = FRAME_H/2 + np.random.uniform(-2,2)
        draw_ellipse(frm, cy, cx, 3*sc, 8*sc, np.random.uniform(32,34), np.random.uniform(-10,10))
        draw_ellipse(frm, cy, cx-9*sc, 2*sc, 2*sc, np.random.uniform(34,36))
        draw_ellipse(frm, cy, cx+9*sc, 2.5*sc, 5*sc, np.random.uniform(30,32))
    else:
        cx = FRAME_W/2 + np.random.uniform(-5,5)
        hcy = np.random.uniform(3,7)*sc
        tcy = hcy + 5*sc; lcy = tcy + 7*sc
        draw_ellipse(frm, hcy, cx, 2*sc, 1.8*sc, np.random.uniform(34,36))
        draw_ellipse(frm, tcy, cx, 4*sc, 2.5*sc, np.random.uniform(32,34))
        if pose == "arms_up":
            draw_ellipse(frm, hcy, cx-4*sc, 1*sc, 2.5*sc, np.random.uniform(33,35), 60)
            draw_ellipse(frm, hcy, cx+4*sc, 1*sc, 2.5*sc, np.random.uniform(33,35), -60)
        elif pose == "sit":
            draw_ellipse(frm, tcy, cx-3.5*sc, 3*sc, 1*sc, np.random.uniform(33,35), 15)
            draw_ellipse(frm, tcy, cx+3.5*sc, 3*sc, 1*sc, np.random.uniform(33,35), -15)
            draw_ellipse(frm, tcy+4*sc, cx-2*sc, 2*sc, 1.2*sc, np.random.uniform(30,32))
            draw_ellipse(frm, tcy+4*sc, cx+2*sc, 2*sc, 1.2*sc, np.random.uniform(30,32))
        else:
            draw_ellipse(frm, tcy, cx-4*sc, 3.5*sc, 1*sc, np.random.uniform(33,35), 10)
            draw_ellipse(frm, tcy, cx+4*sc, 3.5*sc, 1*sc, np.random.uniform(33,35), -10)
            draw_ellipse(frm, lcy, cx-1.5*sc, 3.5*sc, 1*sc, np.random.uniform(30,32))
            draw_ellipse(frm, lcy, cx+1.5*sc, 3.5*sc, 1*sc, np.random.uniform(30,32))
    return frm + _noise(frm.shape, np.random.uniform(0.3,0.8))

def make_human_torso():
    frm = new_frame(); sc = np.random.uniform(0.9,1.4)
    cx = FRAME_W/2+np.random.uniform(-4,4); cy = FRAME_H/2+np.random.uniform(-3,3)
    draw_ellipse(frm, cy, cx, 6*sc, 4*sc, np.random.uniform(32,34), np.random.uniform(-15,15))
    return frm + _noise(frm.shape, np.random.uniform(0.3,0.7))

def make_human_head():
    frm = new_frame(); sc = np.random.uniform(0.8,1.3)
    cx = FRAME_W/2+np.random.uniform(-6,6); cy = FRAME_H/2+np.random.uniform(-5,5)
    draw_ellipse(frm, cy, cx, 3.5*sc, 3*sc, np.random.uniform(34,36))
    draw_ellipse(frm, cy+3.5*sc, cx, 1.5*sc, 1.2*sc, np.random.uniform(33,35))
    return frm + _noise(frm.shape, np.random.uniform(0.3,0.6))

def make_human_hands():
    frm = new_frame(); sc = np.random.uniform(0.8,1.2); n = random.choice([1,2])
    spots = random.sample([(FRAME_H*0.3,FRAME_W*0.25),(FRAME_H*0.3,FRAME_W*0.75),
                           (FRAME_H*0.7,FRAME_W*0.3),(FRAME_H*0.5,FRAME_W*0.5)], k=n)
    for cy,cx in spots:
        draw_ellipse(frm, cy+np.random.uniform(-2,2), cx+np.random.uniform(-2,2),
                     2*sc, 1.5*sc, np.random.uniform(33,35), np.random.uniform(-30,30))
    return frm + _noise(frm.shape, np.random.uniform(0.3,0.7))

def make_human_feet():
    frm = new_frame(); sc = np.random.uniform(0.8,1.2)
    cx = FRAME_W/2+np.random.uniform(-3,3); cy = FRAME_H-np.random.uniform(3,6)
    draw_ellipse(frm, cy, cx-2.5*sc, 2*sc, 1.5*sc, np.random.uniform(30,33), np.random.uniform(-10,10))
    draw_ellipse(frm, cy, cx+2.5*sc, 2*sc, 1.5*sc, np.random.uniform(30,33), np.random.uniform(-10,10))
    return frm + _noise(frm.shape, np.random.uniform(0.3,0.6))

def make_human_partial():
    frm = new_frame(); sc = np.random.uniform(0.8,1.3)
    edge = random.choice(["left","right","top","bottom"])
    if edge=="left":    cx,cy = np.random.uniform(-1,4),FRAME_H/2+np.random.uniform(-4,4)
    elif edge=="right": cx,cy = np.random.uniform(FRAME_W-4,FRAME_W+1),FRAME_H/2+np.random.uniform(-4,4)
    elif edge=="top":   cx,cy = FRAME_W/2+np.random.uniform(-5,5),np.random.uniform(-2,4)
    else:               cx,cy = FRAME_W/2+np.random.uniform(-5,5),np.random.uniform(FRAME_H-4,FRAME_H+1)
    draw_ellipse(frm, cy, cx, 4*sc, 2.5*sc, np.random.uniform(32,34))
    draw_ellipse(frm, cy-5*sc, cx, 2*sc, 1.8*sc, np.random.uniform(34,36))
    return frm + _noise(frm.shape, np.random.uniform(0.3,0.8))

print("Human generators ready.")


In [ ]:
# Part 2 — Hot object and background generators
def make_hot_object():
    frm = new_frame()
    obj = random.choice(["laptop","cup","heater","iron","phone","radiator"])
    cx = np.random.uniform(6, FRAME_W-6); cy = np.random.uniform(5, FRAME_H-5)
    if obj=="laptop":
        w,h,t = np.random.uniform(7,11),np.random.uniform(2,4),np.random.uniform(35,45)
        draw_rect(frm, cy-h, cx-w, cy+h, cx+w, t)
        draw_rect(frm, cy+h-1, cx-w/2, cy+h+1, cx+w/2, t+5)
    elif obj=="cup":
        r = np.random.uniform(1.5,3.0)
        draw_ellipse(frm, cy, cx, r, r*0.7, np.random.uniform(40,55))
    elif obj=="heater":
        w,h,t = np.random.uniform(3,6),np.random.uniform(6,10),np.random.uniform(45,62)
        cx = 3+w if random.choice([True,False]) else FRAME_W-3-w
        draw_rect(frm, cy-h, cx-w, cy+h, cx+w, t)
    elif obj=="iron":
        draw_ellipse(frm, cy, cx, np.random.uniform(1.5,2.5), np.random.uniform(3,5),
                     np.random.uniform(50,65), np.random.uniform(-20,20))
    elif obj=="phone":
        w,h,t = np.random.uniform(1,2),np.random.uniform(3,5),np.random.uniform(35,45)
        draw_rect(frm, cy-h, cx-w, cy+h, cx+w, t)
    elif obj=="radiator":
        w,t = np.random.uniform(8,14),np.random.uniform(40,55)
        row = random.choice([1,2,FRAME_H-3,FRAME_H-2])
        draw_rect(frm, row, cx-w, row+2, cx+w, t)
    return frm + _noise(frm.shape, np.random.uniform(0.3,0.8))

def make_background():
    frm = new_frame(); bg = frm[0,0]
    variant = random.choice(["uniform","warm_wall","sun_patch","gradient"])
    if variant=="warm_wall":
        wall = random.choice(["top","bottom","left","right"])
        warmth = np.random.uniform(1,4)
        if wall=="top":
            for r in range(min(6,FRAME_H)): frm[r,:] += warmth*(1-r/6)
        elif wall=="bottom":
            for r in range(min(6,FRAME_H)): frm[FRAME_H-1-r,:] += warmth*(1-r/6)
        elif wall=="left":
            for c in range(min(6,FRAME_W)): frm[:,c] += warmth*(1-c/6)
        else:
            for c in range(min(6,FRAME_W)): frm[:,FRAME_W-1-c] += warmth*(1-c/6)
    elif variant=="sun_patch":
        cx = np.random.uniform(4,FRAME_W-4); cy = np.random.uniform(3,FRAME_H-3)
        draw_ellipse(frm, cy, cx, np.random.uniform(2,5), np.random.uniform(3,7), bg+np.random.uniform(1,3))
    elif variant=="gradient":
        delta = np.random.uniform(1,3)
        for r in range(FRAME_H): frm[r,:] += delta*r/FRAME_H
    return frm + _noise(frm.shape, np.random.uniform(0.2,0.5))

print("Object and background generators ready.")


In [ ]:
# Part 2 — Generate synthetic dataset
N_PER_HUMAN = 500   # 6 classes x 500 = 3 000 human frames
N_OBJ       = 1000
N_BG        = 1000

generators = [
    (make_human_full,    0, N_PER_HUMAN),
    (make_human_partial, 1, N_PER_HUMAN),
    (make_human_head,    2, N_PER_HUMAN),
    (make_human_torso,   3, N_PER_HUMAN),
    (make_human_hands,   4, N_PER_HUMAN),
    (make_human_feet,    5, N_PER_HUMAN),
    (make_hot_object,    6, N_OBJ),
    (make_background,    7, N_BG),
]

frames_list, labels_list = [], []
for fn, cls, n in generators:
    print(f"  {CLASS_NAMES[cls]}: {n}", end=" ")
    for _ in range(n):
        frames_list.append(fn()); labels_list.append(cls)
    print("done")

X_syn = np.stack(frames_list).astype(np.float32)
y_syn = np.array(labels_list, dtype=np.int32)
np.save(DATA_DIR/"X_syn.npy", X_syn); np.save(DATA_DIR/"y_syn.npy", y_syn)
print(f"Dataset: X={X_syn.shape}  y={y_syn.shape}")
print({CLASS_NAMES[i]: int((y_syn==i).sum()) for i in range(8)})


In [ ]:
# Visualise 25 sample frames
fig, axes = plt.subplots(5, 5, figsize=(13, 10))
idxs = np.random.choice(len(X_syn), 25, replace=False)
for ax, idx in zip(axes.flat, idxs):
    im = ax.imshow(X_syn[idx], cmap="inferno", vmin=23, vmax=50, aspect="auto")
    ax.set_title(CLASS_NAMES[y_syn[idx]], fontsize=8); ax.axis("off")
fig.suptitle("Synthetic Thermal Frames — Sample Grid", fontsize=13)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.5, label="C")
plt.tight_layout()
plt.savefig(PLOT_DIR/"sample_grid.png", dpi=120, bbox_inches="tight"); plt.show()
print("Grid saved.")


---
## Part 3 — Real Data Integration

In [ ]:
# Part 3 — Clone repository
import subprocess
REPO_DIR = Path("/content/Project_Pi")
if not REPO_DIR.exists():
    r = subprocess.run(["git","clone","--depth","1",
        "https://github.com/kupalmananalsal-hub/Project_Pi", str(REPO_DIR)],
        capture_output=True, text=True)
    print("Cloned OK" if r.returncode==0 else f"Clone failed: {r.stderr[:200]}")
else:
    print("Repository already present.")


In [ ]:
# Part 3 — Load real thermal frames
real_frames, real_labels = [], []

def resize_frame(arr):
    if arr.ndim == 3: arr = arr.mean(axis=-1)
    if arr.shape == (FRAME_H, FRAME_W): return arr.astype(np.float32)
    return zoom(arr, (FRAME_H/arr.shape[0], FRAME_W/arr.shape[1]), order=1).astype(np.float32)

def load_npy(folder, label):
    n = 0
    for p in sorted(folder.glob("**/*.npy")):
        try:
            a = np.load(p, allow_pickle=False)
            if a.ndim == 2:
                real_frames.append(resize_frame(a)); real_labels.append(label); n += 1
            elif a.ndim == 3 and a.shape[-1] not in (1,3,4):
                for f in a:
                    real_frames.append(resize_frame(f)); real_labels.append(label); n += 1
        except Exception:
            pass
    return n

def load_csv(folder, label):
    import pandas as pd; n = 0
    for p in sorted(folder.glob("**/*.csv")):
        try:
            df = pd.read_csv(p, header=None)
            a = df.values.astype(np.float32)
            if a.shape[1] == FRAME_H * FRAME_W:
                for row in a:
                    real_frames.append(row.reshape(FRAME_H, FRAME_W))
                    real_labels.append(label); n += 1
        except Exception:
            pass
    return n

DATASETS = [
    (REPO_DIR/"dataset/thermal/raw/thermo_presence",    0),
    (REPO_DIR/"dataset/thermal/raw/mldetection",        1),
    (REPO_DIR/"dataset/thermal/raw/skku_thermal_human", 0),
    (REPO_DIR/"dataset/thermal/raw/yolov8_thermal",     0),
    (REPO_DIR/"dataset/thermal",                        0),
]

total_real = 0
for folder, lbl in DATASETS:
    if not folder.exists():
        print(f"Skipping (missing): {folder.name}"); continue
    n = load_npy(folder, lbl) + load_csv(folder, lbl)
    print(f"  {folder.name}: {n} frames"); total_real += n
print(f"Total real frames loaded: {total_real}")


In [ ]:
# Merge synthetic + real, normalise
if real_frames:
    X_real = np.stack(real_frames).astype(np.float32)
    y_real = np.array(real_labels, dtype=np.int32)
    X_all  = np.concatenate([X_syn, X_real])
    y_all  = np.concatenate([y_syn, y_real])
    print(f"{len(X_syn)} synthetic + {len(X_real)} real = {len(X_all)} total")
else:
    X_all, y_all = X_syn.copy(), y_syn.copy()
    print("Synthetic data only.")

# Normalise: clip to [10,80] C then scale to [0,1]
X_norm  = ((np.clip(X_all, 10, 80) - 10) / 70).astype(np.float32)
X_model = X_norm[..., np.newaxis]   # (N, 24, 32, 1) for Conv2D
print("X_model shape:", X_model.shape)
print("Value range  : [", X_model.min().round(3), ",", X_model.max().round(3), "]")


---
## Part 4 — Model Architecture

In [ ]:
# Part 4 — Model architecture
def build_model(nc=NUM_CLASSES):
    inp = keras.Input(shape=(FRAME_H, FRAME_W, 1), name="thermal_input")
    x = layers.Conv2D(16,(3,3),padding="same",activation="relu",name="conv1")(inp)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.MaxPooling2D((2,2),name="pool1")(x)
    x = layers.Conv2D(32,(3,3),padding="same",activation="relu",name="conv2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.MaxPooling2D((2,2),name="pool2")(x)
    x = layers.Conv2D(64,(3,3),padding="same",activation="relu",name="conv3")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(64,activation="relu",name="dense1")(x)
    x = layers.Dropout(0.3,name="dropout")(x)
    out = layers.Dense(nc,activation="softmax",name="predictions")(x)
    return keras.Model(inp, out, name="thermal_shape_cnn")

model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


---
## Part 5 — Training

In [ ]:
# Part 5 — Train / Val / Test split (70 / 15 / 15)
present = [i for i in range(NUM_CLASSES) if (y_all==i).sum() >= 2]
mask = np.isin(y_all, present)
Xf, yf = X_model[mask], y_all[mask]

X_tv, X_test, y_tv, y_test = train_test_split(
    Xf, yf, test_size=0.15, random_state=RANDOM_SEED, stratify=yf)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.15/0.85, random_state=RANDOM_SEED, stratify=y_tv)

print(f"Train : {len(X_train)}")
print(f"Val   : {len(X_val)}")
print(f"Test  : {len(X_test)}")


In [ ]:
# Part 5 — Training
BEST = str(MODEL_DIR/"best_model.keras")

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=5,
        restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint(
        BEST, monitor="val_accuracy", save_best_only=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30, batch_size=64,
    callbacks=callbacks, verbose=1,
)
print("Best model ->", BEST)


In [ ]:
# Training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history.history["accuracy"],     label="train acc")
ax1.plot(history.history["val_accuracy"], label="val acc")
ax1.set_title("Accuracy"); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(history.history["loss"],     label="train loss")
ax2.plot(history.history["val_loss"], label="val loss")
ax2.set_title("Loss"); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.suptitle("Training History"); plt.tight_layout()
plt.savefig(PLOT_DIR/"training_history.png", dpi=120, bbox_inches="tight"); plt.show()


---
## Part 6 — Evaluation

In [ ]:
# Part 6 — Evaluation
best_model = keras.models.load_model(BEST)
probs  = best_model.predict(X_test, batch_size=64, verbose=0)
y_pred = np.argmax(probs, axis=1)

present_cls   = sorted(np.unique(np.concatenate([y_test, y_pred])))
present_names = [CLASS_NAMES[i] for i in present_cls]

print("=" * 60)
print(classification_report(y_test, y_pred,
      labels=present_cls, target_names=present_names, zero_division=0))
test_acc   = float(np.mean(y_pred == y_test))
test_loss_v = best_model.evaluate(X_test, y_test, batch_size=64, verbose=0)
print("Test accuracy:", round(test_acc, 4), "  Test loss:", round(test_loss_v[0], 4))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=present_cls)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(present_names))); ax.set_yticks(range(len(present_names)))
ax.set_xticklabels(present_names, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(present_names, fontsize=9)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Test Set")
for i in range(len(present_names)):
    for j in range(len(present_names)):
        color = "white" if cm[i,j] > cm.max()*0.55 else "black"
        ax.text(j, i, str(cm[i,j]), ha="center", va="center", fontsize=8, color=color)
plt.colorbar(im, ax=ax, shrink=0.8); plt.tight_layout()
plt.savefig(PLOT_DIR/"confusion_matrix.png", dpi=120, bbox_inches="tight"); plt.show()


In [ ]:
# FPR analysis and binary ROC
y_bin      = (y_test < 6).astype(int)
h_prob     = probs[:, :6].sum(axis=1)
y_bin_pred = (y_pred < 6).astype(int)

obj_mask = y_test == 6; bg_mask = y_test == 7
fpr_obj = float((y_bin_pred[obj_mask]==1).sum()/obj_mask.sum()) if obj_mask.sum()>0 else float("nan")
fpr_bg  = float((y_bin_pred[bg_mask]==1).sum()/bg_mask.sum())  if bg_mask.sum()>0  else float("nan")
print(f"FPR hot_object : {fpr_obj:.3f}  ({fpr_obj*100:.1f}% falsely classified as human)")
print(f"FPR background : {fpr_bg:.3f}  ({fpr_bg*100:.1f}% falsely classified as human)")

fpr_r, tpr_r, _ = roc_curve(y_bin, h_prob)
roc_auc = auc(fpr_r, tpr_r)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_r, tpr_r, color="darkorange", lw=2, label=f"ROC AUC={roc_auc:.3f}")
ax.plot([0,1],[0,1], "navy", lw=1, linestyle="--", label="Random")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC — Human vs Non-Human (Binary)")
ax.legend(loc="lower right"); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR/"roc_curve.png", dpi=120, bbox_inches="tight"); plt.show()
print("AUC:", round(roc_auc, 4))


In [ ]:
# Save metrics
report_dict = classification_report(y_test, y_pred, labels=present_cls,
    target_names=present_names, output_dict=True, zero_division=0)
metrics = {
    "test_accuracy"   : test_acc,
    "test_loss"       : float(test_loss_v[0]),
    "roc_auc"         : float(roc_auc),
    "fpr_hot_object"  : fpr_obj,
    "fpr_background"  : fpr_bg,
    "n_train"         : len(X_train),
    "n_val"           : len(X_val),
    "n_test"          : len(X_test),
    "n_real_frames"   : int(total_real),
    "class_names"     : CLASS_NAMES,
    "classification_report": report_dict,
}
with open(MODEL_DIR/"eval_metrics.json","w") as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print(json.dumps({k:v for k,v in metrics.items() if k!="classification_report"}, indent=2))


---
## Part 7 — TFLite Export

In [ ]:
# Part 7 — TFLite export (dynamic-range quantisation)
TFLITE = MODEL_DIR/"thermal_human_shape.tflite"
LABELS = MODEL_DIR/"thermal_human_shape_labels.json"

conv = tf.lite.TFLiteConverter.from_keras_model(best_model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
tfl  = conv.convert()

with open(TFLITE,"wb") as f: f.write(tfl)
with open(LABELS,"w")  as f: json.dump(CLASS_NAMES, f, indent=2)

print(f"TFLite saved: {TFLITE}  ({len(tfl)/1024:.1f} KB)")
print(f"Labels saved: {LABELS}")


In [ ]:
# Verify TFLite model
interp = tf.lite.Interpreter(model_path=str(TFLITE))
interp.allocate_tensors()
ind  = interp.get_input_details()[0]
outd = interp.get_output_details()[0]
print("Input  shape:", ind["shape"],  " dtype:", ind["dtype"])
print("Output shape:", outd["shape"], " dtype:", outd["dtype"])

n_ok = 0; n = min(50, len(X_test))
for i in range(n):
    s = X_test[i:i+1].astype(ind["dtype"])
    interp.set_tensor(ind["index"], s); interp.invoke()
    if int(np.argmax(interp.get_tensor(outd["index"]))) == int(y_test[i]):
        n_ok += 1
print(f"TFLite accuracy on {n} samples: {n_ok/n:.3f}")


In [ ]:
# Download output files
from google.colab import files
print("Downloading output files...")
files.download(str(TFLITE))
files.download(str(LABELS))
files.download(str(MODEL_DIR/"eval_metrics.json"))


---
## Summary

| File | Description |
|:-----|:------------|
| `thermal_human_shape.tflite` | Quantised model for Pi |
| `thermal_human_shape_labels.json` | Class index mapping |
| `eval_metrics.json` | Full test-set metrics |

### Pi Deployment Snippet

```python
import tflite_runtime.interpreter as tflite, numpy as np, json

interp = tflite.Interpreter("thermal_human_shape.tflite")
interp.allocate_tensors()
labels = json.load(open("thermal_human_shape_labels.json"))

def predict(frame_24x32):
    x = ((np.clip(frame_24x32, 10, 80) - 10) / 70).astype(np.float32)
    x = x[np.newaxis, ..., np.newaxis]
    interp.set_tensor(interp.get_input_details()[0]["index"], x)
    interp.invoke()
    p = interp.get_tensor(interp.get_output_details()[0]["index"])[0]
    return labels[int(np.argmax(p))], float(p.max())

HUMAN = {"human_full","human_partial","human_head","human_torso","human_hands","human_feet"}
label, conf = predict(your_24x32_frame)
human_detected = label in HUMAN and conf >= 0.65
```
